# Golden Dataset Evaluation
Compares trained model checkpoints on a held-out Golden Dataset - images sourced from google (24 of each produce type, 12 rotten; 12 healthy) independently from the training data to give an honest, out-of-distribution accuracy estimate.

Each model was trained under different dataset / augmentation  conditions; the golden set is fixed across all comparisons so the results are directly comparable.


| Model label | Dataset | Augmentation | Notes |
|---|---|---|---|
| Dirty | Full raw dataset (~29k images) | Yes | Includes near-duplicates and static pre-augmented images |
| No dups+aug | No duplicates | No | --- |
| No dups | No duplicates | No | Baseline clean dataset |

# EXPERIMENTS
### Success criteria
- ID val is over-saturated so selection is based on:
    
    `OOD validation accuracy (primary) => OOD AUC-ROC (Secondary) => OOD ECE (Tertiary).`

### Steps
1. Identify best dataset variation using EfficientNet STL baseline (deduplication, pre-augmentation)

    ***Deduplication was won by ECE tiebreake***
2. Identify best STL architecture at fixed optimizer (AdamW): EfficientNet vs Swin vs MaxViT
3. On winning arch, MTL vs STL with unified stopping criterion (primary-task loss)

4. On winning arch, architectural ablations:
    - freeze / partial-freeze / finetune
    - pretrained / random-init
    - class-weighted / unweighted loss
5. Augmentation ablation on best config from (4)
6. Post-hoc: temperature scaling on val > OOD ECE check

In [53]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from torchvision.models import get_model, get_weight
from torch.utils.data import DataLoader
from utils.dataset import ProduceDataset
from pathlib import Path
from safetensors.torch import load_file
sys.path.append("..")
from experiment_configs import task_2_config as experiments
from experiment_configs import task_2_config_final as experiments_f
from utils.mtl_model import MultiTaskClassifier
from sklearn.metrics import roc_curve, roc_auc_score, brier_score_loss, precision_recall_fscore_support,confusion_matrix
import numpy as np
from plotly.subplots import make_subplots

GOLDEN_PATH =  Path(".") / ".." / "golden_dataset" 
NUM_HEALTH_CLASSES = 2
CLASS_NAMES = ["Healthy", "Rotten"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _load_state_dict(path):
    if path.endswith(".safetensors"):
        return load_file(path, device=str(device))
    ckpt = torch.load(path, weights_only=False, map_location=device)
    return ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt


def load_model(exp, path, num_produce_classes):
    """Rebuild architecture from exp config, then load weights."""
    pretrained_weights = get_weight(exp.weight_string)
    base = get_model(exp.architecture, weights=None)
    if exp.is_mtl:
    # MTL: wrap in MultiTaskClassifier (matches saved state_dict keys)
        model = MultiTaskClassifier(base, num_produce_classes=num_produce_classes,
                                    num_health_classes=NUM_HEALTH_CLASSES)
    else:
        # STL: replace final layer using head_attr pattern
        head_attr = "classifier" if hasattr(base, "classifier") else "head"
        head = getattr(base, head_attr)
        if isinstance(head, nn.Sequential):
            head[-1] = nn.Linear(head[-1].in_features, NUM_HEALTH_CLASSES)
        elif isinstance(head, nn.Linear):
            setattr(base, head_attr, nn.Linear(head.in_features, NUM_HEALTH_CLASSES))
        model = base

    model.load_state_dict(_load_state_dict(path))
    return model.to(device).eval(), pretrained_weights.transforms()


def evaluate(model, transforms, is_mtl):
    dataset = ProduceDataset(dataset_root_dir=GOLDEN_PATH, transform=transforms)
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    all_preds, all_labels, all_probs, all_p_rotten = [], [], [], []
    with torch.no_grad():
        for x, y_health, _ in loader:
            x, y_health = x.to(device), y_health.to(device)
            health_out = model(x)[0] if is_mtl else model(x)
            probs = torch.softmax(health_out, dim=1)
            preds = health_out.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y_health.cpu().tolist())
            all_probs.extend(probs.max(dim=1).values.cpu().tolist())
            all_p_rotten.extend(probs[:, 1].cpu().tolist())   # prob of positive (rotten) class

    return pd.DataFrame({
        "produce":    [Path(p).parent.name.split("__")[0] for p in dataset.image_paths],
        "true_label": all_labels,
        "pred":       all_preds,
        "correct":    [p == l for p, l in zip(all_preds, all_labels)],
        "confidence": all_probs,
        "p_rotten":   all_p_rotten,
    })

def compute_ece(labels, preds, confidences, n_bins=10):
    labels, preds, confidences = map(np.asarray, (labels, preds, confidences))
    edges = np.linspace(0, 1, n_bins + 1)
    ece, bins = 0.0, []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (confidences >= lo) & (confidences < hi) if i < n_bins - 1 else (confidences >= lo) & (confidences <= hi)
        if mask.sum() == 0:
            bins.append({"conf": (lo + hi) / 2, "acc": None, "count": 0})
            continue
        bin_conf = confidences[mask].mean()
        bin_acc = (preds[mask] == labels[mask]).mean()
        bins.append({"conf": bin_conf, "acc": bin_acc, "count": int(mask.sum())})
        ece += (mask.sum() / len(confidences)) * abs(bin_acc - bin_conf)
    return ece, bins


In [54]:
# Define models to evaluate

MODELS = {
    # DETERMINE BEST DATASET VARIANT
    "Dirty":              (experiments.EX2_EFFICIENTNET_FINETUNE,         "1_select_dataset/EX2_dirty_efficientnet_finetune20260417190029.pth"),
    "Dedup_no_aug":       (experiments.EX2_EFFICIENTNET_FINETUNE,         "1_select_dataset/EX2_dedup_no_aug_EFFICIENTNET_FINETUNE_20260421113156.safetensors"),
    "Dedup_old_aug":      (experiments.EX2_EFFICIENTNET_FINETUNE,         "1_select_dataset/EX2_dedup_EFFICIENTNET_FINETUNE_20260421123836.safetensors"),
    "Dedup_new_aug_7":    (experiments.EX10_EFFICIENTNET_FINETUNE_AUG,    "1_select_dataset/EX10_EFFICIENTNET_FINETUNE_AUG_20260421211057.safetensors"),
    "Dedup_new_aug_10":   (experiments.EX11_EFFICIENTNET_FINETUNE_AUG,    "1_select_dataset/EX11_EFFICIENTNET_FINETUNE_AUG_20260421221546.safetensors"),
    "Dedup_new_aug_13":   (experiments.EX12_EFFICIENTNET_FINETUNE_AUG,    "1_select_dataset/EX12_EFFICIENTNET_FINETUNE_AUG_20260421210924.safetensors"),
    # DETERMINE BEST ARCHITECTURE
    "EffNet_s":             (experiments_f.EX1_EFFICIENTNET_FINETUNE,       "2_select_arch/EX1_EFFICIENTNET_FINETUNE_20260422022739.safetensors"),
    "EffNet_B4":            (experiments_f.EX1T_EFFICIENTNET_FINETUNE,      "2_select_arch/EX1T_EFFICIENTNET_FINETUNE_20260422032622.safetensors"),
    "Swin_s":               (experiments_f.EX2_SWIN_FINETUNE,               "2_select_arch/EX2_SWIN_FINETUNE_20260422100511.safetensors"),
    "Swin_s_43":               (experiments_f.EX2_SWIN_FINETUNE,               "2_select_arch/EX2_SWIN_FINETUNE_43_20260424013509.safetensors"),
    "Swin_s_44":               (experiments_f.EX2_SWIN_FINETUNE,               "best/EX2_SWIN_FINETUNE_44_20260424015207.safetensors"),
    "Swin_s_45":               (experiments_f.EX2_SWIN_FINETUNE,               "2_select_arch/EX2_SWIN_FINETUNE_45_20260424020906.safetensors"),
    "Swin_t":               (experiments_f.EX2T_SWIN_FINETUNE,              "2_select_arch/EX2T_SWIN_FINETUNE_20260422044721.safetensors"),
    "MaxViT":               (experiments_f.EX3_MAXVIT_FINETUNE,             "2_select_arch/EX3_MAXVIT_FINETUNE_20260422022737.safetensors"),
    # MTL Weighting experiment
    "ENET_90_mtl":          (experiments_f.EX4a_EFFICIENTNET_FINETUNE_MTL,  "3_mtl_comp/EX4a_EFFICIENTNET_FINETUNE_MTL_20260422113149.safetensors"),
    "ENET_75_mtl":          (experiments_f.EX4b_EFFICIENTNET_FINETUNE_MTL,  "3_mtl_comp/EX4b_EFFICIENTNET_FINETUNE_MTL_20260422145627.safetensors"),
    "ENET_50_mtl":          (experiments_f.EX4c_EFFICIENTNET_FINETUNE_MTL,  "3_mtl_comp/EX4c_EFFICIENTNET_FINETUNE_MTL_20260422141950.safetensors"),
    "MaxViT_90_mtl":        (experiments_f.EX5a_MAXVIT_FINETUNE_MTL,        "3_mtl_comp/EX5a_MAXVIT_FINETUNE_MTL_20260422163620.safetensors"),
    "MaxViT_75_mtl":        (experiments_f.EX5b_MAXVIT_FINETUNE_MTL,        "3_mtl_comp/EX5b_MAXVIT_FINETUNE_MTL_20260422112303.safetensors"),
    "MaxViT_50_mtl":        (experiments_f.EX5c_MAXVIT_FINETUNE_MTL,        "3_mtl_comp/EX5c_MAXVIT_FINETUNE_MTL_20260422125321.safetensors"),
    "swin_90_mtl":          (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "3_mtl_comp/EX6a_SWIN_FINETUNE_MTL_20260422201021.safetensors"), # Best
    "swin_75_mtl":          (experiments_f.EX6b_SWIN_FINETUNE_MTL,          "3_mtl_comp/EX6b_SWIN_FINETUNE_MTL_20260422203742.safetensors"),
    "swin_50_mtl":          (experiments_f.EX6c_SWIN_FINETUNE_MTL,          "3_mtl_comp/EX6c_SWIN_FINETUNE_MTL_20260422201057.safetensors"),
    # Best Ablations    
    "swin_freeze":          (experiments_f.EX7a_SWIN_MTL_FREEZE,            "4_ablations/EX7a_SWIN_MTL_FREEZE_20260422225028.safetensors"),                
    "swin_scratch":         (experiments_f.EX7b_SWIN_MTL_SCRATCH,           "4_ablations/EX7b_SWIN_MTL_SCRATCH_20260422231839.safetensors"),
    "swin_unweighted":      (experiments_f.EX7c_SWIN_MTL_UNWEIGHTED,        "4_ablations/EX7c_SWIN_MTL_UNWEIGHTED_20260422225054.safetensors"),
    # Variance tests
    "swin_90_mtl_seed43":       (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "5_variance_test/EX6a_SWIN_FINETUNE_MTL_43_20260423035309.safetensors"),
    "swin_90_mtl_seed44":       (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "5_variance_test/EX6a_SWIN_FINETUNE_MTL_44_20260423050145.safetensors"),
    "swin_90_mtl_seed45":       (experiments_f.EX6a_SWIN_FINETUNE_MTL,          "5_variance_test/EX6a_SWIN_FINETUNE_MTL_45_20260423023125.safetensors"),
    # Augmentation + variance
    "swin_best_aug_seed1":       (experiments_f.EX8_SWIN_FINETUNE_MTL_AUG,          "6_augmentation/EX8_SWIN_FINETUNE_MTL_AUG_42_20260423121851.safetensors"),
    "swin_best_aug_seed2":       (experiments_f.EX8_SWIN_FINETUNE_MTL_AUG,          "6_augmentation/EX8_SWIN_FINETUNE_MTL_AUG_43_20260423133517.safetensors"),
    "swin_best_aug_seed3":       (experiments_f.EX8_SWIN_FINETUNE_MTL_AUG,          "6_augmentation/EX8_SWIN_FINETUNE_MTL_AUG_45_20260423121913.safetensors"),
#     # Low augmentation (spatial only)
    "swin_best_low_aug":                     (experiments_f.EX8a_SWIN_FINETUNE_MTL_AUG,         "6_augmentation/EX8a_SWIN_FINETUNE_MTL_AUG_20260423234629.safetensors" )
}

# Get produce count for MTL
TRAINING_DATA_DIR = Path(".") / "data" / "Fruit_And_Vegetable_Diseases_Dataset_no_identical_no_aug"
ds = ProduceDataset(dataset_root_dir=TRAINING_DATA_DIR)
num_produce_classes = ds.num_produce_types

import pickle
from pathlib import Path

# ── Config: name the current eval dataset so caches don't collide ────────────
EVAL_DATASET_NAME = "golden_1"        # change per run: "golden_2", "golden_3", ...
CACHE_DIR = Path("runs/eval") / EVAL_DATASET_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Set True to force a full re-evaluation (ignores existing cache files)
FORCE_REEVAL = False

# ── Run evaluation with per-model caching ───────────────────────────────────
results = {}
for name, (exp, path) in MODELS.items():
    cache_path = CACHE_DIR / f"{name}.pkl"
    print(cache_path)

    if cache_path.exists() and not FORCE_REEVAL:
        with open(cache_path, "rb") as f:
            results[name] = pickle.load(f)
        print(f"Loaded cached: {name}  ({results[name]['correct'].mean():.4f})")
        continue

    print(f"Evaluating: {name} ({exp.display_name})  "
          f"(arch={exp.architecture}, mtl={exp.is_mtl})")
    model, transforms = load_model(exp, f"models/{path}", num_produce_classes)
    df = evaluate(model, transforms, exp.is_mtl)
    results[name] = df

    with open(cache_path, "wb") as f:
        pickle.dump(df, f)
    print(f"  Overall: {df['correct'].mean():.4f}  (cached -> {cache_path})")

# Optional: also save a single combined file for convenience
with open(CACHE_DIR / "_all.pkl", "wb") as f:
    pickle.dump(results, f)
print(f"\nSaved combined results: {CACHE_DIR / '_all.pkl'}")



runs/eval/golden_1/Dirty.pkl
Loaded cached: Dirty  (0.8408)
runs/eval/golden_1/Dedup_no_aug.pkl
Loaded cached: Dedup_no_aug  (0.8715)
runs/eval/golden_1/Dedup_old_aug.pkl
Loaded cached: Dedup_old_aug  (0.8799)
runs/eval/golden_1/Dedup_new_aug_7.pkl
Loaded cached: Dedup_new_aug_7  (0.8715)
runs/eval/golden_1/Dedup_new_aug_10.pkl
Loaded cached: Dedup_new_aug_10  (0.8128)
runs/eval/golden_1/Dedup_new_aug_13.pkl
Loaded cached: Dedup_new_aug_13  (0.8464)
runs/eval/golden_1/EffNet_s.pkl
Loaded cached: EffNet_s  (0.9134)
runs/eval/golden_1/EffNet_B4.pkl
Loaded cached: EffNet_B4  (0.9022)
runs/eval/golden_1/Swin_s.pkl
Loaded cached: Swin_s  (0.9246)
runs/eval/golden_1/Swin_s_43.pkl
Loaded cached: Swin_s_43  (0.9218)
runs/eval/golden_1/Swin_s_44.pkl
Loaded cached: Swin_s_44  (0.9413)
runs/eval/golden_1/Swin_s_45.pkl
Loaded cached: Swin_s_45  (0.9330)
runs/eval/golden_1/Swin_t.pkl
Loaded cached: Swin_t  (0.9330)
runs/eval/golden_1/MaxViT.pkl
Loaded cached: MaxViT  (0.9302)
runs/eval/golden_1/ENE

In [55]:

# ── Compute all metrics first ─────────────────────────────────────────────────
metrics = {}
rows = []
for name, df in results.items():
    fpr, tpr, _ = roc_curve(df["true_label"], df["p_rotten"])
    auc = roc_auc_score(df["true_label"], df["p_rotten"])
    ece, bins = compute_ece(df["true_label"], df["pred"], df["confidence"])
    prec, rec, f1, _ = precision_recall_fscore_support(
        df["true_label"], df["pred"], average="binary", pos_label=1, zero_division=0
    )
    metrics[name] = {"fpr": fpr, "tpr": tpr, "auc": auc, "ece": ece, "bins": bins}
    rows.append({
        "Model":       name,
        "Accuracy":    df["correct"].mean(),
        "AUC-ROC":     auc,
        "ECE":         ece,
        "Brier":       brier_score_loss(df["true_label"], df["p_rotten"]),
        # "Mean conf.":  df["confidence"].mean(),
        "Precision":   prec,
        "Recall":      rec,
        "F1":          f1,
        "N":           len(df),
    })

# ── Summary table (before figures) ────────────────────────────────────────────
summary = pd.DataFrame(rows).set_index("Model")
higher_better = ["Accuracy", "AUC-ROC", "F1", "Precision", "Recall"]
lower_better  = ["ECE", "Brier"]

def rank_highlight(col, ascending):
    """Color top 3 distinct values: dark > medium > light green."""
    ranks = col.rank(ascending=ascending, method='dense')
    colors = {
        1: 'background-color:#4C9B36; color:black; weight:bold;',  
        2: 'background-color:#97D586 ; color:black',             
        3: 'background-color:#B4E1A8; color:black',       
    }
    return [colors.get(int(r), '') for r in ranks]

display(summary.style
    .format("{:.4f}", subset=summary.columns.difference(["N"]))
    .apply(lambda c: rank_highlight(c, ascending=False), subset=higher_better)
    .apply(lambda c: rank_highlight(c, ascending=True),  subset=lower_better)
)

# ── Overall accuracy bar chart ─────────────────────────────────────────────────
overall = summary[["Accuracy"]].reset_index()
fig1 = px.bar(overall, x="Model", y="Accuracy", text_auto=".3f",
              title="Overall accuracy — golden dataset",
              color="Model", range_y=[0.5, 1.0])
fig1.update_traces(textposition="outside")
fig1.show()

# ── ROC curves ────────────────────────────────────────────────────────────────
fig_roc = go.Figure()
for name, m in metrics.items():
    fig_roc.add_trace(go.Scatter(x=m["fpr"], y=m["tpr"], mode="lines",
                                 name=f"{name} (AUC={m['auc']:.3f})"))
fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                             line=dict(dash="dash", color="gray"), name="Random"))
fig_roc.update_layout(title="ROC curves — golden dataset",
                      xaxis_title="False positive rate", yaxis_title="True positive rate",
                      height=500, width=700)
fig_roc.show()

# ── Reliability diagram ───────────────────────────────────────────────────────
# Reliability diagram — equal-mass (quantile) bins + marker size ∝ sample count
# Fixes the jaggedness from sparse equal-width bins; legend click still toggles each trace
N_BINS_VIS = 10
fig_ece = go.Figure()
for name, df in results.items():
    conf = df["confidence"].values
    correct = df["correct"].values.astype(int)
    edges = np.quantile(conf, np.linspace(0, 1, N_BINS_VIS + 1))
    edges[0], edges[-1] = 0.0, 1.0
    xs, ys, counts = [], [], []
    for i in range(N_BINS_VIS):
        lo, hi = edges[i], edges[i + 1]
        m = (conf >= lo) & (conf < hi) if i < N_BINS_VIS - 1 else (conf >= lo) & (conf <= hi)
        if m.sum() < 2:
            continue
        xs.append(float(conf[m].mean()))
        ys.append(float(correct[m].mean()))
        counts.append(int(m.sum()))
    fig_ece.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines+markers",
        marker=dict(size=[6 + float(np.sqrt(c)) * 1.5 for c in counts]),
        customdata=counts,
        hovertemplate="conf=%{x:.3f}<br>acc=%{y:.3f}<br>n=%{customdata}<extra></extra>",
        name=f"{name} (ECE={metrics[name]['ece']:.3f})",
    ))
fig_ece.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    line=dict(dash="dash", color="gray"), name="Perfect calibration"
))
fig_ece.update_layout(
    title="Reliability diagram (equal-mass bins, marker size ∝ sample count)",
    xaxis_title="Confidence", yaxis_title="Accuracy",
    xaxis=dict(range=[0.4, 1.0]), yaxis=dict(range=[0, 1]),
    height=550, width=900,
    legend=dict(itemclick="toggle", itemdoubleclick="toggleothers"),
)
fig_ece.show()


# ── Per-category grouped bar chart ────────────────────────────────────────────
# per_cat = []
# for name, df in results.items():
#     cat_acc = df.groupby("produce")["correct"].mean().reset_index()
#     cat_acc.columns = ["produce", "accuracy"]
#     cat_acc["model"] = name
#     per_cat.append(cat_acc)

# per_cat_df = pd.concat(per_cat)

# fig2 = px.bar(per_cat_df, x="produce", y="accuracy", color="model",
#               barmode="group", title="Per-category accuracy - golden dataset",
#               range_y=[0, 1.0], text_auto=".2f")
# fig2.update_layout(xaxis_tickangle=-45, height=500)
# fig2.show()

# ── Heatmap ───────────────────────────────────────────────────────────────────
# pivot = per_cat_df.pivot(index="model", columns="produce", values="accuracy")

# fig3 = px.imshow(pivot, text_auto=".2f", aspect="auto",
#                  color_continuous_scale="RdYlGn", range_color=[0.5, 1.0],
#                  title="Accuracy heatmap — model vs category")
# fig3.show()

# OOD golden-set accuracy - task-critical, this is what the system will actually face
# OOD AUC-ROC - robust to class imbalance, threshold-free. Great tiebreaker when accuracies are within 1-2pp
# OOD ECE/brier - calibration matters because ripeness grading downstream reads softmax confidence
# 
# 1. Identify best dataset variation using EfficientNet STL baseline (deduplication, pre-augmentation)
#    Deduplication (old aug) has wins over no aug; however, deduplicaiton + new aug has superior performance.
#    =DEDUPLICATION (NO AUG) WINS=
# 2. Identify best STL architecture at fixed optimizer (AdamW): EfficientNet vs Swin vs MaxViT
#    MAXVIT WINS BY ACC, ECE/BRIER (notably) & F1. Keep Enet_v2 for CNN comparison.
# 3. On winning arch, MTL vs STL with unified stopping criterion (primary-task loss)
# 4. On winning arch, architectural ablations:
#     - freeze / partial-freeze / finetune
#     - pretrained / random-init
#     - class-weighted / unweighted loss
# 5. Augmentation ablation on best config from (4)
# 6. Post-hoc: temperature scaling on val > OOD ECE check


,Accuracy,AUC-ROC,ECE,Brier,Precision,Recall,F1,N
Model,,,,,,,,
Dirty,0.8408,0.9188,0.0845,0.1252,0.8105,0.8800,0.8438,358
Dedup_no_aug,0.8715,0.9505,0.1107,0.1134,0.8413,0.9086,0.8736,358
Dedup_old_aug,0.8799,0.9583,0.0425,0.0872,0.8708,0.8857,0.8782,358
Dedup_new_aug_7,0.8715,0.9501,0.0858,0.1027,0.8772,0.8571,0.8671,358
Dedup_new_aug_10,0.8128,0.9263,0.1086,0.1345,0.8600,0.7371,0.7938,358
Dedup_new_aug_13,0.8464,0.9403,0.0966,0.1096,0.8093,0.8971,0.8509,358
EffNet_s,0.9134,0.9802,0.0593,0.0704,0.9138,0.9086,0.9112,358
EffNet_B4,0.9022,0.9617,0.0593,0.0808,0.9070,0.8914,0.8991,358
Swin_s,0.9246,0.9840,0.0602,0.0668,0.8814,0.9771,0.9268,358


In [ ]:
def load_all_eval(root=Path("runs/eval")):
    results_by_dataset = {}
    for ds_dir in root.iterdir():
        if not ds_dir.is_dir():
            continue
        all_file = ds_dir / "_all.pkl"
        if all_file.exists():
            with open(all_file, "rb") as f:
                results_by_dataset[ds_dir.name] = pickle.load(f)
        else:
            # Fallback: load individual model files
            results_by_dataset[ds_dir.name] = {}
            for model_file in ds_dir.glob("*.pkl"):
                if model_file.stem.startswith("_"):
                    continue
                with open(model_file, "rb") as f:
                    results_by_dataset[ds_dir.name][model_file.stem] = pickle.load(f)
    return results_by_dataset

results_by_dataset = load_all_eval()
print(f"Loaded: {list(results_by_dataset.keys())}")
for ds_name, res in results_by_dataset.items():
    print(f"  {ds_name}: {len(res)} models")

# ── Helper: compute the summary for one dataset ──────────────────────────────
def build_summary(results_dict):
    """results_dict: {model_name: df_with_cols true_label/p_rotten/pred/confidence/correct}"""
    metrics, rows = {}, []
    for name, df in results_dict.items():
        fpr, tpr, _ = roc_curve(df["true_label"], df["p_rotten"])
        auc = roc_auc_score(df["true_label"], df["p_rotten"])
        ece, bins = compute_ece(df["true_label"], df["pred"], df["confidence"])
        prec, rec, f1, _ = precision_recall_fscore_support(
            df["true_label"], df["pred"], average="binary", pos_label=1, zero_division=0
        )
        metrics[name] = {"fpr": fpr, "tpr": tpr, "auc": auc, "ece": ece, "bins": bins}
        rows.append({
            "Model": name,
            "Accuracy": df["correct"].mean(),
            "AUC-ROC": auc,
            "ECE": ece,
            "Brier": brier_score_loss(df["true_label"], df["p_rotten"]),
            "Mean conf.": df["confidence"].mean(),
            "Precision": prec, "Recall": rec, "F1": f1,
            "N": len(df),
        })
    return pd.DataFrame(rows).set_index("Model"), metrics


# ── Load all three datasets' results ─────────────────────────────────────────
# results_by_dataset[ds_name] is a {model_name: df} dict, same shape as before

summaries  = {}   # {ds_name: summary_df}
metrics_by = {}   # {ds_name: metrics_dict}
for ds_name, res in results_by_dataset.items():
    summaries[ds_name], metrics_by[ds_name] = build_summary(res)
    
# ── Optional: filter to specific models. Set to None to show all. ────────────
MODEL_FILTER = ["EffNet_s", "MaxViT", "Swin_s", "Swin_t", "swin_90_mtl"]   # or None

def _filter(summary):
    if MODEL_FILTER is None:
        return summary
    keep = summary.index.intersection(MODEL_FILTER)
    return summary.loc[keep]

# ── Per-dataset tables (now respect the filter) ──────────────────────────────
for ds_name, summary in summaries.items():
    print(f"\n=== {ds_name} ===")
    display(
        _filter(summary).style
            .format("{:.4f}", subset=summary.columns.difference(["N"]))
            .apply(lambda c: rank_highlight(c, ascending=False), subset=higher_better)
            .apply(lambda c: rank_highlight(c, ascending=True),  subset=lower_better)
    )

# ── Combined accuracy bar chart (filter applied, ordering preserved) ─────────
long = []
for ds_name, summary in summaries.items():
    sub = _filter(summary)
    for model_name, row in sub.iterrows():
        long.append({"Model": model_name, "Dataset": ds_name, "Accuracy": row["Accuracy"]})
long_df = pd.DataFrame(long)

# Order by MODEL_FILTER if given, else by golden_1 accuracy
if MODEL_FILTER:
    model_order = [m for m in MODEL_FILTER if m in long_df["Model"].unique()]
else:
    model_order = summaries["golden_1"].sort_values("Accuracy", ascending=False).index.tolist()

fig_combined = px.bar(
    long_df, x="Model", y="Accuracy", color="Dataset",
    barmode="group", text_auto=".3f",
    title="Accuracy across golden datasets",
    range_y=[0.4, 1.0],
    category_orders={"Model": model_order},
)

# Add this line to center the title perfectly
fig_combined.update_layout(title_x=0.5)
fig_combined.update_traces(textposition="outside", textfont_size=9)
fig_combined.update_layout(
    height=500,
    width=max(700, 60 * len(model_order) * 3),  # ~60px per bar × 3 datasets per group
    xaxis_tickangle=-30,
    legend_title="Dataset",
)
fig_combined.show()


from pathlib import Path

OUT_DIR = Path("../report/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LIGHT_GREEN = [
    [0.00, "#ffffff"], [0.40, "#e8f5e9"],
    [0.75, "#c8e6c9"], [1.00, "#81c784"],
]
DISPLAY_COLS = ["Accuracy", "AUC-ROC", "ECE", "Brier", "Precision", "Recall", "F1"]
LOWER_BETTER = {"ECE", "Brier"}


def metric_heatmap_aggregated(summaries, model_filter, lower_better,
                              cols=DISPLAY_COLS, title=None, save_name=None):
    """Mean across all golden splits, per (model, metric)."""
    stacked = [_filter(s)[list(cols)] for s in summaries.values() if not _filter(s).empty]
    means = pd.concat(stacked).groupby(level=0).mean()
    if model_filter:
        means = means.loc[[m for m in model_filter if m in means.index]]

    z = means.copy().astype(float)
    for col in cols:
        v = means[col].astype(float)
        if col in lower_better:
            z[col] = (v.max() - v) / (v.max() - v.min() + 1e-8)
        else:
            z[col] = (v - v.min()) / (v.max() - v.min() + 1e-8)

    fig = go.Figure(data=go.Heatmap(
        z=z.values, x=means.columns.tolist(), y=means.index.tolist(),
        colorscale=LIGHT_GREEN, showscale=False,
        text=means.values, texttemplate="%{text:.3f}",
        textfont={"size": 12, "color": "black"},
        xgap=2, ygap=2, zmin=0, zmax=1,
    ))
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        height=80 + 45 * len(means),
        margin=dict(l=130, r=20, t=60, b=40),
        plot_bgcolor="white",
        yaxis=dict(autorange="reversed"),
        xaxis=dict(side="top"),
    )
    if save_name:
        fig.write_image(str(OUT_DIR / save_name), scale=2)
    fig.show()
    return fig


metric_heatmap_aggregated(
    summaries, MODEL_FILTER, LOWER_BETTER,
    title=f"Mean performance across {len(summaries)} golden splits",
    save_name="ood_aggregated_heatmap.png",
)
from plotly.subplots import make_subplots

def metric_heatmap_faceted(summaries, model_filter, lower_better,
                           cols=DISPLAY_COLS, title=None, save_name=None):
    ds_names = list(summaries.keys())
    fig = make_subplots(rows=1, cols=len(ds_names),
                        subplot_titles=ds_names,
                        horizontal_spacing=0.04)

    for j, ds in enumerate(ds_names, start=1):
        sub = _filter(summaries[ds])[list(cols)]
        if model_filter:
            sub = sub.loc[[m for m in model_filter if m in sub.index]]
        z = sub.copy().astype(float)
        for col in cols:
            v = sub[col].astype(float)
            if col in lower_better:
                z[col] = (v.max() - v) / (v.max() - v.min() + 1e-8)
            else:
                z[col] = (v - v.min()) / (v.max() - v.min() + 1e-8)

        fig.add_trace(go.Heatmap(
            z=z.values, x=sub.columns.tolist(),
            y=sub.index.tolist() if j == 1 else [""] * len(sub),  # only first panel labels
            colorscale=LIGHT_GREEN, showscale=False,
            text=sub.values, texttemplate="%{text:.3f}",
            textfont={"size": 10, "color": "black"},
            xgap=2, ygap=2, zmin=0, zmax=1,
        ), row=1, col=j)

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        height=80 + 50 * len(_filter(summaries[ds_names[0]])),
        plot_bgcolor="white",
        margin=dict(l=130, r=20, t=80, b=40),
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(side="top", tickangle=-30)
    if save_name:
        fig.write_image(str(OUT_DIR / save_name), scale=2)
    fig.show()
    return fig


metric_heatmap_faceted(
    summaries, MODEL_FILTER, LOWER_BETTER,
    title="Per-split performance across golden datasets",
    save_name="ood_faceted_heatmap.png",
)

Loaded: ['golden_1', 'golden_3', 'golden_2']
  golden_1: 33 models
  golden_3: 33 models
  golden_2: 33 models

=== golden_1 ===


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
EffNet_s,0.9134,0.9802,0.0593,0.0704,0.9699,0.9138,0.9086,0.9112,358
Swin_s,0.9246,0.9840,0.0602,0.0668,0.9819,0.8814,0.9771,0.9268,358
Swin_t,0.9330,0.9840,0.0593,0.0612,0.9818,0.9037,0.9657,0.9337,358
MaxViT,0.9302,0.9848,0.0430,0.0533,0.9732,0.9121,0.9486,0.9300,358
swin_90_mtl,0.9358,0.9825,0.0472,0.0528,0.9808,0.9086,0.9657,0.9363,358



=== golden_3 ===


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
EffNet_s,0.8878,0.9606,0.0746,0.0937,0.9623,0.8889,0.8865,0.8877,740
Swin_s,0.9000,0.9724,0.0694,0.0804,0.9694,0.8700,0.9405,0.9039,740
Swin_t,0.9135,0.9720,0.0655,0.0749,0.9739,0.8864,0.9486,0.9164,740
MaxViT,0.9135,0.9670,0.0460,0.0697,0.9571,0.9250,0.9000,0.9123,740
swin_90_mtl,0.8919,0.9659,0.0858,0.0934,0.9777,0.8662,0.9270,0.8956,740



=== golden_2 ===


,Accuracy,AUC-ROC,ECE,Brier,Mean conf.,Precision,Recall,F1,N
Model,,,,,,,,,
EffNet_s,0.8756,0.9480,0.0880,0.1061,0.9587,0.8778,0.8739,0.8758,442
Swin_s,0.8891,0.9662,0.0720,0.0857,0.9612,0.8681,0.9189,0.8928,442
Swin_t,0.9027,0.9659,0.0701,0.0819,0.9691,0.8745,0.9414,0.9067,442
MaxViT,0.9027,0.9563,0.0476,0.0784,0.9453,0.9324,0.8694,0.8998,442
swin_90_mtl,0.8665,0.9542,0.1137,0.1150,0.9753,0.8410,0.9054,0.8720,442


In [57]:
from pathlib import Path
CANONICAL_DATASET = "golden_1"   # ← edit if you want a different split as canonical
assert CANONICAL_DATASET in summaries, \
    f"'{CANONICAL_DATASET}' not in summaries (have: {list(summaries.keys())})"

canonical = summaries[CANONICAL_DATASET]   # use this everywhere instead of `summary`
TABLES_DIR = Path("../report/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

DISPLAY_COLS = ["Accuracy", "AUC-ROC", "ECE", "Brier", "Precision", "Recall", "F1"]
SHORT_HEADERS = {
    "Accuracy": "Acc", "AUC-ROC": "AUC", "ECE": "ECE", "Brier": "Brier",
    "Precision": "Prec.", "Recall": "Rec.", "F1": "F1",
}

def df_to_latex(df, label, caption, save_name):
    df = df[DISPLAY_COLS].round(3)
    headers = [SHORT_HEADERS.get(c, c) for c in df.columns]

    lines = [r"\begin{table}[htpb]", r"\centering",
             rf"\caption{{{caption}}}",
             rf"\label{{tab:{label}}}",
             r"\begin{tabular}{l" + "c" * len(df.columns) + "}",
             r"\toprule",
             "Model & " + " & ".join(headers) + r" \\",
             r"\midrule"]

    for idx, row in df.iterrows():
        safe = str(idx).replace("_", r"\_")
        lines.append(safe + " & " + " & ".join(f"{v:.3f}" for v in row.values) + r" \\")

    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

    out = TABLES_DIR / save_name
    out.write_text("\n".join(lines))
    print(f"Saved {out}")


# Usage stays the same:

# Ablations
ABLATION_ROWS = ["swin_freeze", "swin_scratch", "swin_unweighted"]
arch_rows = ["EffNet_s", "EffNet_B4", "Swin_s", "Swin_t", "MaxViT"]
df_to_latex(summary.loc[arch_rows],
            label="arch_bakeoff",
            caption="Architecture bakeoff on the golden set (N=740).",
            save_name="arch_bakeoff.tex")

mtl_rows = ["ENET_90_mtl", "ENET_75_mtl", "ENET_50_mtl",
            "MaxViT_90_mtl", "MaxViT_75_mtl", "MaxViT_50_mtl",
            "swin_90_mtl", "swin_75_mtl", "swin_50_mtl"]
df_to_latex(summary.loc[mtl_rows],
            label="mtl_sweep",
            caption="MTL weight sweep on golden set. Primary-task weights 0.5, 0.75, 0.9.",
            save_name="mtl_sweep.tex")

seed_rows = ["Swin_s", "Swin_s_43", "Swin_s_44", "Swin_s_45",
             "swin_90_mtl", "swin_90_mtl_seed43", "swin_90_mtl_seed44", "swin_90_mtl_seed45"]
df_to_latex(summary.loc[seed_rows],
            label="seed_variance",
            caption="Seed variance comparison: Swin STL vs Swin MTL ($w=0.9$).",
            save_name="seed_variance.tex")

Saved ../report/tables/arch_bakeoff.tex
Saved ../report/tables/mtl_sweep.tex
Saved ../report/tables/seed_variance.tex


In [58]:
import plotly.graph_objects as go

# Fill in val ECE from your wandb logs / training-time prints; golden from summary df
data = [
    {"model": "EffNet_s",     "val": 0.007, "golden": summary.loc["EffNet_s", "ECE"]},
    {"model": "MaxViT",       "val": 0.004, "golden": summary.loc["MaxViT", "ECE"]},
    {"model": "Swin_s STL",   "val": 0.005, "golden": summary.loc["Swin_s_44", "ECE"]},
    {"model": "Swin_90 MTL",  "val": 0.005, "golden": summary.loc["swin_90_mtl", "ECE"]},
]

fig = go.Figure()
from pathlib import Path

OUT_DIR = Path("../report/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

fig.write_image(str(OUT_DIR / "ece_dumbbell.png"), scale=2)
# Connecting line for each model
for d in data:
    fig.add_trace(go.Scatter(
        x=[d["val"], d["golden"]],
        y=[d["model"], d["model"]],
        mode="lines",
        line=dict(color="lightgray", width=4),
        showlegend=False,
        hoverinfo="skip",
    ))

# Endpoints
fig.add_trace(go.Scatter(
    x=[d["val"] for d in data],
    y=[d["model"] for d in data],
    mode="markers",
    marker=dict(size=16, color="#1f77b4", line=dict(color="white", width=2)),
    name="Val (in-distribution)",
    hovertemplate="%{y}: ECE = %{x:.3f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=[d["golden"] for d in data],
    y=[d["model"] for d in data],
    mode="markers",
    marker=dict(size=16, color="#d62728", line=dict(color="white", width=2)),
    name="Golden (out-of-distribution)",
    hovertemplate="%{y}: ECE = %{x:.3f}<extra></extra>",
))

# Annotate the multiplicative gap on each row
for d in data:
    ratio = d["golden"] / d["val"]
    fig.add_annotation(
        x=d["golden"], y=d["model"],
        text=f"  {ratio:.0f}×",
        showarrow=False,
        xanchor="left",
        font=dict(size=11, color="#444"),
    )

fig.update_layout(
    title="ECE under distribution shift (val → golden)",
    xaxis_title="Expected Calibration Error",
    yaxis_title="",
    yaxis=dict(autorange="reversed"),  # top model at top
    xaxis=dict(range=[0, max(d["golden"] for d in data) * 1.25]),
    height=350, width=750,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(l=120, r=80, t=60, b=50),
    plot_bgcolor="white",
)
fig.update_xaxes(showgrid=True, gridcolor="#eee")
fig.update_yaxes(showgrid=False)

fig.write_image("../report/figures/ece_dumbbell.png", scale=2)  # if you have kaleido
fig.show()

In [59]:
import plotly.graph_objects as go
import pandas as pd
from pathlib import Path

OUT_DIR = Path("../report/figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROWS = ["EffNet_s", "EffNet_B4", "Swin_s", "Swin_t", "MaxViT"]
LOWER_BETTER = {"ECE", "Brier"}
DISPLAY_COLS = ["Accuracy", "AUC-ROC", "ECE", "Brier", "Precision", "Recall", "F1"]

# Light-green colour scale — top of range is medium green, not near-black
LIGHT_GREEN = [
    [0.00, "#ffffff"],
    [0.40, "#e8f5e9"],
    [0.75, "#c8e6c9"],
    [1.00, "#81c784"],
]

def metric_heatmap(df, lower_better, title, save_name=None,
                   colorscale=LIGHT_GREEN, height=320):
    """Annotated heatmap. Per-column normalised; darker = better."""
    z = df.copy().astype(float)
    for col in df.columns:
        v = df[col].astype(float)
        if col in lower_better:
            z[col] = (v.max() - v) / (v.max() - v.min() + 1e-8)
        else:
            z[col] = (v - v.min()) / (v.max() - v.min() + 1e-8)

    fig = go.Figure(data=go.Heatmap(
        z=z.values,
        x=df.columns.tolist(),
        y=df.index.tolist(),
        colorscale=colorscale,
        showscale=False,
        text=df.values,
        texttemplate="%{text:.3f}",
        textfont={"size": 12, "color": "black"},
        xgap=2, ygap=2,
        zmin=0, zmax=1,
    ))
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        height=height,
        margin=dict(l=130, r=20, t=60, b=40),
        plot_bgcolor="white",
        yaxis=dict(autorange="reversed"),
        xaxis=dict(side="top"),
    )
    if save_name:
        fig.write_image(str(OUT_DIR / save_name), scale=2)
    fig.show()
    return fig


# ── Verification: confirm we're pulling from the golden-set summary ──────────
arch_df = summary.loc[ROWS, DISPLAY_COLS]

n_values = summary.loc[ROWS, "N"].unique()
print("Source check — these values should match your summary table:")
print(arch_df.round(3))
print(f"\nN per row: {n_values}")

assert len(n_values) == 1, \
    f"N mismatch — rows were evaluated on different sets: {n_values}"
N = int(n_values[0])

# ── Render ───────────────────────────────────────────────────────────────────
metric_heatmap(
    arch_df,
    lower_better=LOWER_BETTER,
    title=f"Architecture bakeoff on the golden set (N={N})",
    save_name="arch_bakeoff_heatmap.png",
)


Source check — these values should match your summary table:
           Accuracy  AUC-ROC    ECE  Brier  Precision  Recall     F1
Model                                                               
EffNet_s      0.876    0.948  0.088  0.106      0.878   0.874  0.876
EffNet_B4     0.860    0.933  0.093  0.117      0.904   0.806  0.852
Swin_s        0.889    0.966  0.072  0.086      0.868   0.919  0.893
Swin_t        0.903    0.966  0.070  0.082      0.874   0.941  0.907
MaxViT        0.903    0.956  0.048  0.078      0.932   0.869  0.900

N per row: [442]


In [60]:
def metric_facets(df, lower_better, title, save_name=None,
                  cols_per_row=4, height_per_row=240):
    metrics = df.columns.tolist()
    n = len(metrics)
    n_cols = min(n, cols_per_row)
    n_rows = (n + n_cols - 1) // n_cols

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=metrics,
        vertical_spacing=0.18, horizontal_spacing=0.08,
    )

    for i, metric in enumerate(metrics):
        r, c = i // n_cols + 1, i % n_cols + 1
        v = df[metric].values
        best = v.argmin() if metric in lower_better else v.argmax()
        colors = ["#2E6F40" if j == best else "#888888" for j in range(len(v))]

        fig.add_trace(
            go.Bar(
                x=df.index, y=v,
                marker_color=colors,
                text=[f"{x:.3f}" for x in v],
                textposition="outside",
                showlegend=False,
                cliponaxis=False,
            ),
            row=r, col=c,
        )
        # Tighten y-range so differences are visible
        margin = (v.max() - v.min()) * 0.5 or 0.02
        fig.update_yaxes(range=[v.min() - margin, v.max() + margin * 1.6], row=r, col=c)

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        height=height_per_row * n_rows,
        plot_bgcolor="white",
    )
    fig.update_xaxes(tickangle=-30)
    if save_name:
        fig.write_image(str(OUT_DIR / save_name), scale=2)
    fig.show()
    return fig


# Same data, different visualisation
metric_facets(
    arch_df,
    lower_better=LOWER_BETTER,
    title="Architecture bakeoff (golden set, N=740). Best per metric in green.",
    save_name="arch_bakeoff_facets.png",
)

## 2. Cross-Model Comparison
High-level comparison across all models: overall accuracy, per-category breakdown, and an accuracy heatmap. Key observation: the dirty dataset (largest volume) outperforms cleaner subsets, suggesting data volume dominates over data cleanliness for this task at current scale.

In [61]:
from pathlib import Path

TABLES_DIR = Path("../report/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

DISPLAY_COLS = ["Accuracy", "AUC-ROC", "ECE", "Brier", "Precision", "Recall", "F1"]
SHORT_HEADERS = {
    "Accuracy": "Acc", "AUC-ROC": "AUC", "ECE": "ECE", "Brier": "Brier",
    "Precision": "Prec.", "Recall": "Rec.", "F1": "F1",
}

def df_to_latex(df, label, caption, save_name):
    df = df[DISPLAY_COLS].round(3)
    headers = [SHORT_HEADERS.get(c, c) for c in df.columns]

    lines = [r"\begin{table}[htpb]", r"\centering",
             rf"\caption{{{caption}}}",
             rf"\label{{tab:{label}}}",
             r"\begin{tabular}{l" + "c" * len(df.columns) + "}",
             r"\toprule",
             "Model & " + " & ".join(headers) + r" \\",
             r"\midrule"]

    for idx, row in df.iterrows():
        safe = str(idx).replace("_", r"\_")
        lines.append(safe + " & " + " & ".join(f"{v:.3f}" for v in row.values) + r" \\")

    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

    out = TABLES_DIR / save_name
    out.write_text("\n".join(lines))
    print(f"Saved {out}")


# Usage stays the same:
arch_rows = ["EffNet_s", "EffNet_B4", "Swin_s", "Swin_t", "MaxViT"]
df_to_latex(summary.loc[arch_rows],
            label="arch_bakeoff",
            caption="Architecture bakeoff on the golden set (N=740).",
            save_name="arch_bakeoff.tex")

mtl_rows = ["ENET_90_mtl", "ENET_75_mtl", "ENET_50_mtl",
            "MaxViT_90_mtl", "MaxViT_75_mtl", "MaxViT_50_mtl",
            "swin_90_mtl", "swin_75_mtl", "swin_50_mtl"]
df_to_latex(summary.loc[mtl_rows],
            label="mtl_sweep",
            caption="MTL weight sweep on golden set. Primary-task weights 0.5, 0.75, 0.9.",
            save_name="mtl_sweep.tex")

seed_rows = ["Swin_s", "Swin_s_43", "Swin_s_44", "Swin_s_45",
             "swin_90_mtl", "swin_90_mtl_seed43", "swin_90_mtl_seed44", "swin_90_mtl_seed45"]
df_to_latex(summary.loc[seed_rows],
            label="seed_variance",
            caption="Seed variance comparison: Swin STL vs Swin MTL ($w=0.9$).",
            save_name="seed_variance.tex")

Saved ../report/tables/arch_bakeoff.tex
Saved ../report/tables/mtl_sweep.tex
Saved ../report/tables/seed_variance.tex


In [62]:
for model_name, df in results.items():
    y_true = df["true_label"].values
    y_pred = df["pred"].values
    conf   = df["confidence"].values

    # ── Precompute all four panels ──
    cm = confusion_matrix(y_true, y_pred)

    categories = sorted(df["produce"].unique())
    rows = []
    for cat in categories:
        sub = df[df["produce"] == cat]
        p, r, f, _ = precision_recall_fscore_support(
            sub["true_label"], sub["pred"], average="binary", zero_division=0)
        rows.append({"Category": cat, "Precision": p, "Recall": r, "F1": f})
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    rows.append({"Category": "MACRO", "Precision": p_mac, "Recall": r_mac, "F1": f_mac})
    prf_df = pd.DataFrame(rows)

    errors = df[~df["correct"]].copy()
    errors["error_type"] = np.where(errors["true_label"] == 0, "H→R", "R→H")
    total = df.groupby("produce").size()
    breakdown = errors.groupby(["produce", "error_type"]).size().unstack(fill_value=0)
    breakdown_pct = (breakdown.div(total, axis=0) * 100).round(1)

    # ── Build 2x2 grid ──
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=("Confusion matrix", "Per-category P / R / F1",
                        "Confidence by outcome",  "Error direction per category (%)"),
        horizontal_spacing=0.12, vertical_spacing=0.15,
        column_widths=[0.35, 0.65],
    )

    # 1. Confusion matrix
    fig.add_trace(go.Heatmap(
        z=cm, x=CLASS_NAMES, y=CLASS_NAMES, colorscale="Blues",
        text=cm, texttemplate="%{text}", showscale=False,
    ), row=1, col=1)

    # 2. P/R/F1 per category (grouped bars)
    for metric, color in [("Precision", "#636EFA"), ("Recall", "#EF553B"), ("F1", "#00CC96")]:
        fig.add_trace(go.Bar(
            name=metric, x=prf_df["Category"], y=prf_df[metric],
            marker_color=color, legendgroup=metric,
        ), row=1, col=2)

    # 3. Confidence distribution — violin (plays nice with grouped barmode elsewhere)
    fig.add_trace(go.Violin(
        x=np.where(df["correct"], "Correct", "Incorrect"),
        y=conf, box_visible=True, meanline_visible=True,
        points="outliers", line_color="#333", fillcolor="#AAB7FF",
        showlegend=False,
    ), row=2, col=1)

    # 4. Error direction
    err_long = breakdown_pct.reset_index().melt(id_vars="produce", var_name="error_type", value_name="value")
    for etype, color in [("H→R", "#EF553B"), ("R→H", "#636EFA")]:
        sub = err_long[err_long["error_type"] == etype]
        fig.add_trace(go.Bar(
            name=etype, x=sub["produce"], y=sub["value"],
            marker_color=color, legendgroup=etype,
        ), row=2, col=2)

    fig.update_layout(
        height=750, width=1300,
        title_text=f"<b>{model_name}</b>",
        barmode="group",
    )
    fig.update_xaxes(tickangle=-45, row=1, col=2)
    fig.update_xaxes(tickangle=-45, row=2, col=2)
    fig.update_yaxes(range=[0, 1.05], row=1, col=2)
    fig.update_yaxes(title_text="Confidence", row=2, col=1)

    fig.show()


In [63]:
from scipy.optimize import minimize_scalar
import torch.nn.functional as F

# ── Temperature scaling calibration ───────────────────────────────────────────
# Calibrate the no-aug model (most overconfident).
# T is found by minimising NLL on the golden set — in practice you'd use a
# separate val set, but this demonstrates the technique on available data.

TARGET_MODEL = "No aug"

def evaluate_with_logits(model, transforms):
    """Same as evaluate() but also returns raw logits for calibration."""
    dataset = ProduceDataset(root_dir=GOLDEN_PATH, transform=transforms)
    loader  = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            all_logits.append(model(imgs).cpu())
            all_labels.extend(labels.tolist())
    return torch.cat(all_logits), torch.tensor(all_labels)

# Reload the target model and collect logits
m, transforms = load_model(MODELS[TARGET_MODEL])
logits, labels = evaluate_with_logits(m, transforms)

# Find T that minimises NLL on the golden set
def nll(T):
    return F.cross_entropy(logits / T, labels).item()

result   = minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded")
best_T   = result.x
print(f"Optimal temperature T = {best_T:.3f}")

# ── Compare confidence distributions before and after ─────────────────────────
def conf_from_logits(logits, T=1.0):
    probs = torch.softmax(logits / T, dim=1)
    return probs.max(dim=1).values.numpy()

preds_orig = logits.argmax(dim=1).numpy()
correct    = (preds_orig == labels.numpy())

conf_before = conf_from_logits(logits, T=1.0)
conf_after  = conf_from_logits(logits, T=best_T)

fig_cal = make_subplots(rows=1, cols=2,
                        subplot_titles=["Before (T=1)", f"After (T={best_T:.2f})"])

for col, conf, title in [(1, conf_before, "Before"), (2, conf_after, "After")]:
    fig_cal.add_trace(go.Histogram(
        x=conf[correct],  name="Correct",   marker_color="#00CC96",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)
    fig_cal.add_trace(go.Histogram(
        x=conf[~correct], name="Incorrect", marker_color="#EF553B",
        opacity=0.7, nbinsx=20, showlegend=(col==1)), row=1, col=col)

fig_cal.update_xaxes(range=[0.4, 1.0])
fig_cal.update_layout(barmode="overlay", title=f"{TARGET_MODEL} — confidence before vs after temperature scaling",
                      height=400)
fig_cal.show()

acc = correct.mean()
print(f"Accuracy before: {acc:.4f}")
print(f"Accuracy after:  {acc:.4f}  (unchanged — argmax is T-invariant)")

KeyError: 'No aug'

## APPENDIX CELLS

In [64]:
import plotly.graph_objects as go
from pathlib import Path

OUT_DIR = Path("../report/figures"); OUT_DIR.mkdir(parents=True, exist_ok=True)

aug_groups = {
    "no aug (4 seeds)":        ["swin_90_mtl", "swin_90_mtl_seed43", "swin_90_mtl_seed44", "swin_90_mtl_seed45"],
    "light aug (H/V flip)":    ["swin_best_low_aug"],
    "sweep aug (3 seeds)":     ["swin_best_aug_seed1", "swin_best_aug_seed2", "swin_best_aug_seed3"],
}

records = []
for label, rows in aug_groups.items():
    sub = canonical.loc[canonical.index.intersection(rows)]
    records.append({
        "config": label,
        "acc_mean": sub["Accuracy"].mean(), "acc_std": sub["Accuracy"].std(ddof=0),
        "ece_mean": sub["ECE"].mean(),      "ece_std": sub["ECE"].std(ddof=0),
        "n": len(sub),
    })

cfgs = [r["config"] for r in records]
fig = go.Figure()
fig.add_trace(go.Bar(name="Accuracy", x=cfgs, y=[r["acc_mean"] for r in records],
    error_y=dict(type="data", array=[r["acc_std"] for r in records]),
    marker_color="#2E6F40", text=[f"{r['acc_mean']:.3f}" for r in records], textposition="outside"))
fig.add_trace(go.Bar(name="ECE", x=cfgs, y=[r["ece_mean"] for r in records],
    error_y=dict(type="data", array=[r["ece_std"] for r in records]),
    marker_color="#B7472A", text=[f"{r['ece_mean']:.3f}" for r in records], textposition="outside"))
fig.update_layout(title="Augmentation comparison (Swin MTL canonical)", barmode="group",
    yaxis_title="Score", height=400, plot_bgcolor="white")
fig.write_image(str(OUT_DIR / "appendix_c_aug.png"), scale=2)
fig.show()


In [65]:
# If you saved the optuna study to a sqlite DB, load it. Otherwise hardcode.
import plotly.graph_objects as go

importances = {"mixup_alpha": 0.96, "cutmix_alpha": 0.04}  # from your sweep output

fig = go.Figure(go.Bar(
    x=list(importances.keys()),
    y=list(importances.values()),
    text=[f"{v:.2f}" for v in importances.values()],
    textposition="outside",
    marker_color="#2E6F40",
))
fig.update_layout(
    title="MixUp/CutMix sweep — parameter importance<br>"
          "<sup>Sweep converged on near-zero alphas; effectively disabled both augmentations.</sup>",
    yaxis_title="Importance (variance share)", yaxis=dict(range=[0, 1]),
    height=350, width=500, plot_bgcolor="white",
)
fig.write_image(str(OUT_DIR / "appendix_d_mixup.png"), scale=2)
fig.show()


In [66]:
ablation_rows = ["swin_90_mtl", "swin_freeze", "swin_scratch", "swin_unweighted"]
labels = ["Baseline (FT)", "FREEZE", "SCRATCH", "UNWEIGHTED"]
sub = canonical.loc[ablation_rows]

fig = go.Figure()
fig.add_trace(go.Bar(name="Accuracy", x=labels, y=sub["Accuracy"],
    text=[f"{v:.3f}" for v in sub["Accuracy"]], textposition="outside",
    marker_color="#2E6F40"))
fig.add_trace(go.Bar(name="F1", x=labels, y=sub["F1"],
    text=[f"{v:.3f}" for v in sub["F1"]], textposition="outside",
    marker_color="#69A067"))
fig.add_trace(go.Bar(name="ECE (lower=better)", x=labels, y=sub["ECE"],
    text=[f"{v:.3f}" for v in sub["ECE"]], textposition="outside",
    marker_color="#B7472A"))
fig.update_layout(
    title="Architectural ablations on Swin MTL (canonical)",
    barmode="group", yaxis_title="Score",
    height=400, plot_bgcolor="white",
)
fig.write_image(str(OUT_DIR / "appendix_e_ablations.png"), scale=2)
fig.show()


In [ ]:
from utils.grade_produce import visualize_bhattacharyya_score
import matplotlib.pyplot as plt

# Three canonical failure-mode images (use your test_data/)
F1_GENERIC_FAIL = "data/test_data/cauliflower.jpg"   # naturally desaturated
F2_SINGLEREF_FAIL = "data/test_data/banana_clean.jpg"  # idealised stock image
F3_DUAL_OK = "data/test_data/strawberry_dim.jpg"       # noisy real photo

# Save the 3 dual-reference visualisations as separate PNGs
for path, name in [
    (F1_GENERIC_FAIL,  "appendix_f_cauliflower.png"),
    (F2_SINGLEREF_FAIL, "appendix_f_banana_stock.png"),
    (F3_DUAL_OK,       "appendix_f_strawberry.png"),
]:
    visualize_bhattacharyya_score(
        path,
        fruit_type="banana",  # or cauliflower / strawberry as appropriate
        healthy_refs=healthy_colour_refs,
        rotten_refs=rotten_refs,
        save_path=str(OUT_DIR / name),
    )
